In [1]:
import pickle
import librosa
import numpy as np
import IPython.display as ipd
from sklearn.model_selection import train_test_split

In [2]:
with open("dataset/dataset.pkl", "rb") as f:
    dataset = pickle.load(f)
    x_files = dataset["gtr"]
    y_files = dataset["ney"]

x_train_files, x_test_files, y_train_files, y_test_files = train_test_split(
    x_files, y_files, test_size=0.2, random_state=42)

print(len(x_train_files), len(x_test_files))

200 50


In [3]:
NEY_SPECTROGRAM_DIR = "dataset/spectrograms/ney/"
GTR_SPECTROGRAM_DIR = "dataset/spectrograms/ac_gtr/"

In [13]:
spectrogram = np.array([
    np.load(GTR_SPECTROGRAM_DIR + x_train_files[0], allow_pickle=True),
    np.load(GTR_SPECTROGRAM_DIR + x_train_files[1], allow_pickle=True)
])

In [15]:
spectrogram = np.expand_dims(spectrogram, axis=3)
spectrogram.shape

(2, 256, 40, 1)

In [17]:
spec_tr = np.transpose(spectrogram, (0, 3, 2, 1))
spec_tr.shape

(2, 1, 40, 256)

spectrogram -> audio

In [2]:
SR = 48000
WINDOW_LENGTH = 10200
FRAME_SIZE = 512
HOP_LENGTH = 256

In [18]:
signal, _ = librosa.load("dataset/ney/00_Ney_C_3.wav", mono=True, sr=SR)
ipd.Audio(signal, rate=SR)

In [19]:
signal.shape

(102000,)

In [20]:
num_windows = int(len(signal) / WINDOW_LENGTH)
spectrograms = []
for i in range(num_windows):
    signal_window = signal[i*WINDOW_LENGTH: (i+1) * WINDOW_LENGTH]
    stft = librosa.stft(signal_window, n_fft=FRAME_SIZE,
                        hop_length=HOP_LENGTH)
    spectrogram = np.abs(stft)
    log_spectrogram = librosa.amplitude_to_db(spectrogram)
    spectrograms.append(log_spectrogram)

In [21]:
len(spectrograms)

10

In [22]:
signals = []
for spec in spectrograms:
    amplitude = librosa.db_to_amplitude(spec)
    signal = librosa.istft(amplitude, n_fft=FRAME_SIZE, hop_length=HOP_LENGTH)
    signals.extend(signal)

In [23]:
len(signals)

99840

In [24]:
ipd.Audio(signals, rate=SR)